# FDRSelector

Keeps every feature whose ANOVA F-test p-value passes the Benjamini-Hochberg
procedure. On average, at most a fraction `alpha` of the kept features are
false discoveries. It picks its own count.

- **Cost:** one F-test per feature. As fast as ANOVAFSelector.
- **Strength:** the count follows a stated error rate instead of a guess.
- **Weakness:** the F-test's blind spots (see ANOVAFSelector), and the count
  can be large on a strong target or zero on a weak one.
- **Modes:** `task='classification'` or `'regression'`. Wraps scikit-learn's
  `SelectFdr`.

In [1]:
import numpy as np
from sklearn.datasets import make_classification

from featsel.selectors import FDRSelector

# 500 features, the first 10 informative (shuffle=False keeps them first).
X, y = make_classification(n_samples=300, n_features=500, n_informative=10,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(10))

## `alpha` sets the error rate

In [2]:
for alpha in (0.2, 0.05, 0.01):
    chosen = set(FDRSelector(alpha=alpha).fit(X, y).get_support(indices=True))
    print(f'alpha={alpha:<5} kept {len(chosen):3d}: '
          f'{len(chosen & truth)} informative, {len(chosen - truth)} false')

alpha=0.2   kept   8: 4 informative, 4 false
alpha=0.05  kept   5: 4 informative, 1 false
alpha=0.01  kept   3: 3 informative, 0 false


## Where it fails

With no signal, nothing passes, and the selector says so instead of handing a
classifier zero features.

In [3]:
rng = np.random.default_rng(0)
X_noise, y_noise = rng.normal(size=(300, 500)), rng.integers(0, 2, 300)
try:
    FDRSelector(alpha=0.05).fit(X_noise, y_noise)
except ValueError as error:
    print(error)

No feature passes the false discovery rate at alpha=0.05. Increase alpha.
